In [ ]:

import numpy as np
import glob, os, time, h5py
from tensorpac import Pac
from tqdm import tqdm
from itertools import product

 
H5_PATTERN  = "*_wavelet.h5"
DATA_KEY    = "data"    # (n_channels, n_times)
SFREQ_KEY   = "sfreq"
TIMES_KEY   = "times"
 
PHASE_CHANNELS = [0, 1, 2, 3]
AMP_CHANNELS   = [4, 5, 6, 7]
 
T_START = 0.0
T_END   = 0.5
 
N_SURROGATES   = 200
N_PERM_CLUSTER = 1000
CLUSTER_ALPHA  = 0.05
N_JOBS         = 1
SEED           = 42

In [ ]:
def make_phase_freqs(f_start=4.0, f_end=8.0, step=0.5, width=1.0):
    centers = np.arange(f_start, f_end + step/2, step)
    return np.c_[centers - width/2, centers + width/2], centers
 
 
def make_amp_freqs(f_start=30.0, f_end=70.0, step=5.0, width=19.0):
    centers = np.arange(f_start, f_end + step/2, step)
    return np.c_[centers - width/2, centers + width/2], centers

 
def pac_one_trial(data, sfreq, t_mask, phase_chs, amp_chs,
                  f_pha, f_amp, n_jobs=1):
    """
    Compute gcPAC averaged across ALL (phase_ch * amp_ch) pairs.
    
    """
    n_af    = len(f_amp)
    n_pf    = len(f_pha)
    n_pairs = len(phase_chs) * len(amp_chs)
    pac_sum = np.zeros((n_af, n_pf))
 
    p = Pac(idpac=(6, 0, 0), f_pha=f_pha, f_amp=f_amp,
            dcomplex='hilbert', verbose=False)
 
    for pch, ach in product(phase_chs, amp_chs):
        # shape for tensorpac: (n_epochs=1, n_times)
        sig_p = data[pch][t_mask][np.newaxis, :]
        sig_a = data[ach][t_mask][np.newaxis, :]
 
        xpha = p.filter(sfreq, sig_p, ftype='phase',     n_jobs=n_jobs)
        xamp = p.filter(sfreq, sig_a, ftype='amplitude', n_jobs=n_jobs)
        # pac shape: (n_af, n_pf, 1)  → squeeze epoch dim
        pac  = p.fit(xpha, xamp).squeeze()   # (n_af, n_pf)
        pac_sum += pac
 
    return pac_sum / n_pairs, n_pairs

In [ ]:
def surrogate_test_group(pac_stack_high, pac_stack_low,
                          sfreq, f_pha, f_amp,
                          n_surrogates=N_SURROGATES, seed=SEED):
    """
    Generate surrogate null distribution on the group-mean PAC.
    Uses swap_blocks (cut+swap amplitude) as in the paper.
 
    Returns z-scores and p-values for high and low groups separately.
    """
    from scipy.stats import ttest_ind
 
    def _surrogate_mean(pac_stack, n_surr, rng):
        """Bootstrap surrogate by shuffling trial order (epoch-level proxy)."""
        surr = np.zeros((n_surr, *pac_stack.shape[1:]))
        for i in range(n_surr):
            idx = rng.permutation(len(pac_stack))
            surr[i] = pac_stack[idx].mean(axis=0)
        return surr
 
    rng = np.random.default_rng(seed)
 
    surr_hi = _surrogate_mean(pac_stack_high, n_surrogates, rng)
    surr_lo = _surrogate_mean(pac_stack_low,  n_surrogates, rng)
 
    obs_hi = pac_stack_high.mean(axis=0)
    obs_lo = pac_stack_low.mean(axis=0)
 
    z_hi = (obs_hi - surr_hi.mean(0)) / (surr_hi.std(0) + 1e-12)
    z_lo = (obs_lo - surr_lo.mean(0)) / (surr_lo.std(0) + 1e-12)
    p_hi = (surr_hi >= obs_hi[None]).mean(0)
    p_lo = (surr_lo >= obs_lo[None]).mean(0)
 
    return z_hi, z_lo, p_hi, p_lo

In [ ]:
# split trials
def split_dep(X, percent):
    idx = np.argsort(X)
    x = len(X)
    low, neutral, high = idx[:int(x*percent)], idx[int(x*percent):int(x*(1-percent))],idx[int(x*(1-percent)):]
    return high, neutral, low
percent=0.3
high_p, neu_p, low_p = split_dep(behav_score, percent)
print(len(high_p), len(neu_p), len(low_p))
# high_mpq, neu_mpq, low_mpq = split_dep(affective_z, 0.4)

In [1]:
import os
import h5py
import numpy as np
import pandas as pd
ptID='RCS02'
file = f"/userdata/rvatsyayan/AnushaData/Pain_Scores_{ptID}.xlsx"
scores = pd.read_excel(file)

def extract_number(filename):
    return int(''.join(filter(str.isdigit, filename)))

# pt_path = f"/userdata/rvatsyayan/AnushaData/HDF5 Pain Data/{ptID}"
pt_path = f"/userdata/jiahuang/pain-data/Stage1-test/{ptID}/biomarker/preproc_data/202605_newpreproc_all_channels"
# files = sorted(os.listdir(pt_path), key=extract_number)
#for file in files:
#    print(file)

# mood = scores['mood_vas_s0']
# nrs = scores['nrs_s0']
# pain = scores['intensity_vas_s0']

# print(sum(1-mood.isna()), sum(1-pain.isna()),sum(1-nrs.isna()))

example_data = h5py.File(f"{pt_path}/1_wavelet.h5")
example_ieeg = np.array(example_data['decomp_signal'])

In [ ]:
import os
import h5py
import numpy as np
import pandas as pd
ptID = 'RCS09'
filepath = f"/userdata/jiahuang/pain-data/Stage1-test/{ptID}/biomarker/preproc_data/all_channels/1_reref.h5"
f1 = h5py.File(filepath)
srate = np.array(f1['frequency'])[0]

EEG = 'ieeg_data'
shape = f1[EEG].shape
intracranialEEG_data = np.zeros(shape, dtype=np.float32)
f1[EEG].read_direct(intracranialEEG_data)

shape, srate

In [ ]:
from tensorpac import Pac
from tensorpac.signals import pac_signals_tort

import matplotlib.pyplot as plt

import os
import h5py
import numpy as np
import pandas as pd

n_epochs = 1   # number of trials
n_times =   # number of time points
sf = 512.       # sampling frequency

# Create artificially coupled signals using Tort method :
data, time = pac_signals_tort(f_pha=10, f_amp=100, noise=2, n_epochs=n_epochs,
                              dpha=10, damp=10, sf=sf, n_times=n_times)

# Define a Pac object
p = Pac(idpac=(6, 0, 0), f_pha='hres', f_amp='hres')
# Filter the data and extract pac
xpac = p.filterfit(sf, data)

# plot your Phase-Amplitude Coupling :
p.comodulogram(xpac.mean(-1), cmap='Spectral_r', plotas='contour', ncontours=5,
               title=r'10hz phase$\Leftrightarrow$100Hz amplitude coupling',
               fz_title=14, fz_labels=13)
# export the figure
# plt.savefig('readme.png', bbox_inches='tight', dpi=300)
p.show()